# 02 — Feature Matrix Construction

**Goal:** Convert 900 × 8-tool raw outputs into a single tidy feature matrix (`data/processed/feature_matrix.parquet`).

**Construction order:**
1. DefenseFinder (defence + anti-defence systems)
2. PADLOC (defence systems)
3. Merge DF + PADLOC via `system_name_map.csv`
4. ResFinder (ARG counts)
5. ICEberg (IME/ICE counts, with BLAST coverage filter)
6. BacMet (HMRG counts, with BLAST coverage filter)
7. ISEScan (IS element counts by family)
8. MLST + metadata (join keys and labels)
9. Derived features: ratios, sparsity filter, ARG burden tertile labels
10. Save to parquet

Each section is explained before the code. Run cells in order — later sections depend on earlier ones.

## Imports and paths

All paths are relative to the notebook's location (`notebooks/`). The `..` prefix navigates up to the project root.

`numpy` is imported but not used until ratio computation — importing at the top avoids hunting for it later.

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

# ── Project layout ──────────────────────────────────────────────────────────
ROOT    = Path("..")                   # project root (one level up from notebooks/)
INTERIM = ROOT / "data" / "interim"   # per-species tool outputs
PROC    = ROOT / "data" / "processed" # final feature matrix lands here
CONFIG  = ROOT / "config"

PROC.mkdir(parents=True, exist_ok=True)  # create processed/ if missing

# ── Six ESKAPE species, matching interim/ subdirectory names ─────────────────
SPECIES = [
    "abaumannii",
    "ecloaceae",
    "efaecium",
    "kpneumoniae",
    "paeruginosa",
    "saureus",
]

# ── Accession normalisation ───────────────────────────────────────────────
# Tool output dirs/files use GCF_XXXXXXXXX_V (underscore before version).
# NCBI canonical form is GCF_XXXXXXXXX.V (dot before version).
# All downstream joins use the canonical dot form.
def norm_acc(s: str) -> str:
    """GCF_000505685_1 → GCF_000505685.1"""
    prefix, version = s.rsplit("_", 1)   # split on LAST underscore only
    return f"{prefix}.{version}"

print("Paths OK")
print(f"  interim : {INTERIM.resolve()}")
print(f"  processed: {PROC.resolve()}")

Paths OK
  interim : /Users/Vicky/Acinetobacter_ML_2/eskape-defence-ml/data/interim
  processed: /Users/Vicky/Acinetobacter_ML_2/eskape-defence-ml/data/processed


## Section 1 — Parse DefenseFinder outputs

### Why this step

DefenseFinder is the primary source for defence system calls. Each genome has its own subdirectory under `interim/{species}/defensefinder/{accession}/`. Inside is `defense_finder_systems.tsv` with one row per detected system **instance** (a genome with three RM systems has three rows).

Key columns we use:
- `subtype` — the specific system name (e.g. `RM_Type_I`, `BREX_I`, `SspBCDE`)
- `activity` — `"Defense"` or `"Antidefense"` (anti-defence is output when `--antidefensefinder` is passed to DefenseFinder)

We parse defence and anti-defence into separate long-form tables. A long-form table has one row per (genome, system) observation — potentially multiple rows for the same genome if it carries multiple instances. We deduplicate later when building presence/absence columns; we keep the raw counts for the count columns.

### Mapping DF subtype names → canonical names

DefenseFinder and PADLOC use different names for the same systems (e.g. DF calls it `SspBCDE`, PADLOC calls it `PT_SspABCD`). The `system_name_map.csv` resolves this. We load it first and build a lookup dictionary: `df_subtype → canonical_name`.

Systems where `source == 'exclude'` are dropped (only `DMS_other`, a catch-all present in 886/900 genomes — uninformative).

In [2]:
# ── Load system name map ─────────────────────────────────────────────────────
name_map = pd.read_csv(CONFIG / "system_name_map.csv")

print(f"system_name_map: {len(name_map)} rows")
print(name_map["source"].value_counts().to_string())
print()

# Build lookup: df_subtype → canonical_name
# Exclude 'exclude' rows (DMS_other catch-all — 886/900 genomes, uninformative).
# Keep 'both', 'df_only', 'df_antidefense' — all have a valid df_subtype entry.
# Note: 'padloc_only' rows have no df_subtype, so they correctly drop out here.
_df_map_rows = name_map[name_map["source"] != "exclude"].dropna(subset=["df_subtype"])
df_subtype_to_canonical = _df_map_rows.set_index("df_subtype")["canonical_name"].to_dict()

print(f"DF subtype → canonical lookup: {len(df_subtype_to_canonical)} entries")
# Spot-check the key systems from the published paper
for key in ["SspBCDE", "Gao_Qat", "RM_Type_I", "BREX_I", "NARP1"]:
    print(f"  {key!r:20s} → {df_subtype_to_canonical.get(key, 'MISSING')}")

system_name_map: 448 rows
source
df_only           150
padloc_only       144
both              113
df_antidefense     40
exclude             1

DF subtype → canonical lookup: 303 entries
  'SspBCDE'            → SspBCDE
  'Gao_Qat'            → Gao_Qat
  'RM_Type_I'          → RM_Type_I
  'BREX_I'             → BREX_I
  'NARP1'              → adf_NARP1


### Parse defence systems (activity == "Defense")

The loop structure is: **species → genome subdirectory → read TSV → tag with genome_id and species**.

Missing or empty TSVs (genomes with zero defence systems) produce zero rows — this is correct, not an error. After the loop we concatenate all species into one long-form table and apply the canonical name mapping.

Unmapped subtypes (not in `system_name_map`) are dropped with a warning. This should not happen if the map was built correctly from the full dataset, but we surface it explicitly rather than silently discarding data.

In [3]:
df_defense_records = []   # accumulates one mini-dataframe per genome
df_missing = []           # tracks missing TSV files (should be zero)

for sp in SPECIES:
    df_dir = INTERIM / sp / "defensefinder"  # e.g. data/interim/abaumannii/defensefinder/

    for genome_dir in sorted(df_dir.iterdir()):  # each subdirectory = one genome
        if not genome_dir.is_dir():
            continue  # skip .DS_Store and other non-directory files

        tsv_path = genome_dir / "defense_finder_systems.tsv"

        if not tsv_path.exists():
            df_missing.append((sp, genome_dir.name))  # log, don't crash
            continue

        raw = pd.read_csv(tsv_path, sep="\t")

        if raw.empty:
            continue  # genome has zero defence systems — valid result

        # Keep defence rows only; anti-defence is parsed in the next cell
        defense_rows = raw[raw["activity"] == "Defense"][["subtype"]].copy()

        if defense_rows.empty:
            continue

        # Tag with canonical genome_id and species
        defense_rows["genome_id"] = norm_acc(genome_dir.name)  # GCF_000505685_1 → GCF_000505685.1
        defense_rows["species"]   = sp

        df_defense_records.append(defense_rows)

# ── Concatenate all species ─────────────────────────────────────────────────
if df_defense_records:
    df_defense_long = pd.concat(df_defense_records, ignore_index=True)
else:
    df_defense_long = pd.DataFrame(columns=["subtype", "genome_id", "species"])

print(f"DefenseFinder defence rows (raw, before name mapping): {len(df_defense_long):,}")
print(f"Missing TSV files: {len(df_missing)}")
if df_missing:
    print("  ", df_missing[:10])

DefenseFinder defence rows (raw, before name mapping): 8,043
Missing TSV files: 0


In [4]:
# ── Apply canonical name mapping ─────────────────────────────────────────────
# Map df_subtype → canonical_name using the lookup built above.
# Rows whose subtype is not in the map get NaN in canonical_name;
# we surface those as warnings, then drop them.
df_defense_long["canonical_name"] = df_defense_long["subtype"].map(df_subtype_to_canonical)

unmapped = df_defense_long[df_defense_long["canonical_name"].isna()]
if not unmapped.empty:
    print(f"WARNING: {len(unmapped)} rows have unmapped subtypes:")
    print(unmapped["subtype"].value_counts().head(20).to_string())
else:
    print("All DF defence subtypes mapped successfully.")

# Drop unmapped rows
df_defense_long = df_defense_long.dropna(subset=["canonical_name"])

print(f"\nRows after mapping: {len(df_defense_long):,}")
print(f"Unique canonical systems detected: {df_defense_long['canonical_name'].nunique()}")
print(f"Unique genomes with >=1 defence system: {df_defense_long['genome_id'].nunique()}")
print()
print("Top 10 most frequent defence systems:")
print(df_defense_long["canonical_name"].value_counts().head(10).to_string())

All DF defence subtypes mapped successfully.

Rows after mapping: 8,043
Unique canonical systems detected: 263
Unique genomes with >=1 defence system: 900

Top 10 most frequent defence systems:
canonical_name
RM_Type_I         876
RM_Type_IV        345
RM_Type_II        290
Gabija            245
AbiE              219
RosmerTA          204
df_Mok_Hok_Sok    189
df_gcu233         189
df_MazEF          170
df_SDIC3          159


### Parse anti-defence systems (activity == "Antidefense")

Same loop, different filter. Anti-defence systems (e.g. `NARP1`, `Ocr`) are phage-encoded proteins that neutralise host defences. They appear in bacterial genomes when carried on prophages or MGEs.

These map to entries in `system_name_map` where `source == 'df_antidefense'`. The lookup dictionary built above already includes these rows.

In [5]:
df_antidefense_records = []

for sp in SPECIES:
    df_dir = INTERIM / sp / "defensefinder"

    for genome_dir in sorted(df_dir.iterdir()):
        if not genome_dir.is_dir():
            continue  # skip .DS_Store

        tsv_path = genome_dir / "defense_finder_systems.tsv"

        if not tsv_path.exists():
            continue

        raw = pd.read_csv(tsv_path, sep="\t")

        if raw.empty:
            continue

        antidefense_rows = raw[raw["activity"] == "Antidefense"][["subtype"]].copy()

        if antidefense_rows.empty:
            continue

        antidefense_rows["genome_id"] = norm_acc(genome_dir.name)
        antidefense_rows["species"]   = sp

        df_antidefense_records.append(antidefense_rows)

if df_antidefense_records:
    df_antidefense_long = pd.concat(df_antidefense_records, ignore_index=True)
else:
    df_antidefense_long = pd.DataFrame(columns=["subtype", "genome_id", "species"])

# Map to canonical names
df_antidefense_long["canonical_name"] = df_antidefense_long["subtype"].map(df_subtype_to_canonical)

unmapped_adf = df_antidefense_long[df_antidefense_long["canonical_name"].isna()]
if not unmapped_adf.empty:
    print(f"WARNING: {len(unmapped_adf)} unmapped anti-defence subtypes:")
    print(unmapped_adf["subtype"].value_counts().head(10).to_string())

df_antidefense_long = df_antidefense_long.dropna(subset=["canonical_name"])

print(f"Anti-defence rows: {len(df_antidefense_long):,}")
print(f"Unique anti-defence systems: {df_antidefense_long['canonical_name'].nunique()}")
print(f"Genomes carrying >=1 anti-defence system: {df_antidefense_long['genome_id'].nunique()}")
print()
print("Top anti-defence systems:")
print(df_antidefense_long["canonical_name"].value_counts().head(10).to_string())

Anti-defence rows: 2,931
Unique anti-defence systems: 40
Genomes carrying >=1 anti-defence system: 795

Top anti-defence systems:
canonical_name
adf_ardb_klca_acric11    541
adf_arda_ardu            418
adf_acriia21             393
adf_psiab                233
adf_Aca_alone            194
adf_ardc                 163
adf_apyc1                152
adf_NARP1                140
adf_Adnd_p0020_p0021     119
adf_acrie9               106


### Section 1 outputs\n\n`df_defense_long` and `df_antidefense_long` are long-form tables — one row per system **instance** per genome. They are not pivoted to wide here.\n\nWhy defer the pivot? Because the wide table needs to merge DefenseFinder and PADLOC calls for the same canonical system (e.g. both tools may detect `AbiC` in the same genome). That merge requires both long-form tables to be in memory simultaneously. Section 3 does this.\n\nThe anti-defence long-form will be pivoted in Section 3 directly (no PADLOC counterpart — PADLOC does not predict anti-defence systems).

In [6]:
# Section 1 summary — print shapes of long-form tables
print(f"df_defense_long    : {len(df_defense_long):,} rows | "
      f"{df_defense_long['genome_id'].nunique()} genomes | "
      f"{df_defense_long['canonical_name'].nunique()} unique systems")

print(f"df_antidefense_long: {len(df_antidefense_long):,} rows | "
      f"{df_antidefense_long['genome_id'].nunique()} genomes | "
      f"{df_antidefense_long['canonical_name'].nunique()} unique anti-defence systems")

print()
print("Top 10 defence systems (by instance count across all genomes):")
print(df_defense_long["canonical_name"].value_counts().head(10).to_string())

print()
print("Top 5 anti-defence systems:")
print(df_antidefense_long["canonical_name"].value_counts().head(5).to_string())

df_defense_long    : 8,043 rows | 900 genomes | 263 unique systems
df_antidefense_long: 2,931 rows | 795 genomes | 40 unique anti-defence systems

Top 10 defence systems (by instance count across all genomes):
canonical_name
RM_Type_I         876
RM_Type_IV        345
RM_Type_II        290
Gabija            245
AbiE              219
RosmerTA          204
df_Mok_Hok_Sok    189
df_gcu233         189
df_MazEF          170
df_SDIC3          159

Top 5 anti-defence systems:
canonical_name
adf_ardb_klca_acric11    541
adf_arda_ardu            418
adf_acriia21             393
adf_psiab                233
adf_Aca_alone            194


### Verify: coverage across all 900 genomes

Every genome that has a DefenseFinder output directory should appear in at least one of the two tables (defense or anti-defence). Genomes with zero systems of either type simply won't appear in either table — they get zero-filled when we join everything later.

This cell checks how many distinct genome IDs appeared in the DF output, and whether that matches the expected 900.

In [7]:
# All genome_ids seen across defence + antidefence
all_df_genomes = set(df_defense_long["genome_id"]) | set(df_antidefense_long["genome_id"])
print(f"Genomes with >=1 DF hit (defence or anti-defence): {len(all_df_genomes)}")

# Build the full 900-genome ID list from the directory structure
all_genome_ids = []
for sp in SPECIES:
    df_dir = INTERIM / sp / "defensefinder"
    for genome_dir in sorted(df_dir.iterdir()):
        if not genome_dir.is_dir():
            continue  # skip .DS_Store
        all_genome_ids.append(norm_acc(genome_dir.name))

all_genome_ids = sorted(set(all_genome_ids))
print(f"Total genome IDs from directory scan:               {len(all_genome_ids)}")

zero_hit_genomes = set(all_genome_ids) - all_df_genomes
print(f"Genomes with zero DF hits (pure zeros):             {len(zero_hit_genomes)}")

Genomes with >=1 DF hit (defence or anti-defence): 900
Total genome IDs from directory scan:               900
Genomes with zero DF hits (pure zeros):             0


## Section 2 — Parse PADLOC outputs

### Why this step and what is different from DefenseFinder

PADLOC and DefenseFinder use the same underlying biology but different HMM databases and system boundary definitions. Running both tools and merging the results is standard practice in the field — systems detected by both tools are higher-confidence calls than those detected by only one.

The critical structural difference from DF: **PADLOC outputs one row per gene (protein HMM hit), not one row per system instance.** A single AbiL system contains two proteins (AbiLi and AbiLii), so it produces two rows, both with `system.number = 1`. If a genome has two AbiL copies, all four gene rows appear — two with `system.number = 1` and two with `system.number = 2`.

To get the instance count correctly, we deduplicate on `(system, system.number)` before building the long-form. This collapses multi-gene rows for the same system instance into one row.

### Mapping PADLOC system names → canonical names

Same principle as Section 1: `system_name_map.csv` contains a `padloc_system` column that maps PADLOC names to `canonical_name`. For `padloc_only` systems, the canonical name carries no tool prefix (e.g. `AVAST_II`, not `padloc_AVAST_II`). This is by design — the canonical name is the biology, not the tool.

After mapping, the canonical names in `padloc_long` and `df_defense_long` use the **same** vocabulary, so Section 3's merge is a straightforward union on `canonical_name`.

In [8]:
# ── Build padloc_system → canonical_name lookup ──────────────────────────────
# Exclude 'exclude' rows and rows with no padloc_system entry (df_only, df_antidefense).
_padloc_map_rows = name_map[name_map["source"] != "exclude"].dropna(subset=["padloc_system"])
padloc_system_to_canonical = _padloc_map_rows.set_index("padloc_system")["canonical_name"].to_dict()

print(f"PADLOC system → canonical lookup: {len(padloc_system_to_canonical)} entries")
for key in ["AbiC", "PT_SspABCD", "qatABCD", "brex_type_I", "AVAST_type_II"]:
    print(f"  {key!r:22s} → {padloc_system_to_canonical.get(key, 'MISSING')}")

PADLOC system → canonical lookup: 257 entries
  'AbiC'                 → AbiC
  'PT_SspABCD'           → SspBCDE
  'qatABCD'              → Gao_Qat
  'brex_type_I'          → BREX_I
  'AVAST_type_II'        → AVAST_II


In [9]:
padloc_records = []
padloc_missing  = []   # files absent entirely
padloc_unmapped_subtypes = []  # system names not in name_map

for sp in SPECIES:
    padloc_dir = INTERIM / sp / "padloc"  # flat directory — one CSV per genome

    for csv_file in sorted(padloc_dir.iterdir()):
        # PADLOC files: GCF_000505685_1_padloc.csv  (and .gff, .faa, .domtblout)
        if not csv_file.name.endswith("_padloc.csv"):
            continue

        # Reconstruct genome_id: strip the trailing '_padloc' before normalising
        stem = csv_file.stem.removesuffix("_padloc")   # GCF_000505685_1_padloc → GCF_000505685_1
        genome_id = norm_acc(stem)                       # GCF_000505685_1 → GCF_000505685.1

        raw = pd.read_csv(csv_file)

        if raw.empty:
            continue  # genome has zero PADLOC hits — valid

        # Deduplicate to one row per system INSTANCE:
        # system.number identifies distinct occurrences of the same system type within a genome.
        # Multiple gene rows with the same system.number belong to the same instance.
        instances = raw.drop_duplicates(subset=["system", "system.number"])[["system"]].copy()

        if instances.empty:
            continue

        instances["genome_id"] = genome_id
        instances["species"]   = sp

        padloc_records.append(instances)

# ── Concatenate ──────────────────────────────────────────────────────────────
if padloc_records:
    padloc_long_raw = pd.concat(padloc_records, ignore_index=True)
else:
    padloc_long_raw = pd.DataFrame(columns=["system", "genome_id", "species"])

print(f"PADLOC rows after instance deduplication (raw): {len(padloc_long_raw):,}")
print(f"Unique PADLOC system names seen: {padloc_long_raw['system'].nunique()}")
print(f"Genomes with >=1 PADLOC hit: {padloc_long_raw['genome_id'].nunique()}")

PADLOC rows after instance deduplication (raw): 10,772
Unique PADLOC system names seen: 258
Genomes with >=1 PADLOC hit: 900


In [10]:
# ── Apply canonical name mapping ─────────────────────────────────────────────
padloc_long_raw["canonical_name"] = padloc_long_raw["system"].map(padloc_system_to_canonical)

unmapped_padloc = padloc_long_raw[padloc_long_raw["canonical_name"].isna()]
if not unmapped_padloc.empty:
    print(f"WARNING: {len(unmapped_padloc)} rows with unmapped PADLOC system names:")
    print(unmapped_padloc["system"].value_counts().head(20).to_string())
    print()
else:
    print("All PADLOC system names mapped successfully.")

padloc_long = padloc_long_raw.dropna(subset=["canonical_name"]).copy()

print(f"Rows after mapping: {len(padloc_long):,}")
print(f"Unique canonical systems from PADLOC: {padloc_long['canonical_name'].nunique()}")
print(f"Genomes with >=1 mapped PADLOC system: {padloc_long['genome_id'].nunique()}")
print()
print("Top 10 PADLOC systems (by instance count):")
print(padloc_long["canonical_name"].value_counts().head(10).to_string())
print()

# ── Cross-tool sanity check ───────────────────────────────────────────────────
# Systems in BOTH tools' long-form tables: these are the 'both' rows in name_map.
# They should show up in both df_defense_long and padloc_long.
df_systems   = set(df_defense_long["canonical_name"].unique())
padloc_sys   = set(padloc_long["canonical_name"].unique())
in_both      = df_systems & padloc_sys
df_only_seen = df_systems - padloc_sys
padloc_only_seen = padloc_sys - df_systems

print(f"Systems seen by BOTH tools:    {len(in_both)}")
print(f"Systems seen by DF only:       {len(df_only_seen)}")
print(f"Systems seen by PADLOC only:   {len(padloc_only_seen)}")

system
DMS_other    886

Rows after mapping: 9,886
Unique canonical systems from PADLOC: 257
Genomes with >=1 mapped PADLOC system: 900

Top 10 PADLOC systems (by instance count):
canonical_name
padloc_PDC-S07    716
RM_Type_I         585
padloc_SoFic      410
AbiE              384
Gabija            336
padloc_PDC-S04    335
padloc_PDC-S12    322
PD-T4-6           322
VSPR              310
Mokosh_TypeII     224

Systems seen by BOTH tools:    113
Systems seen by DF only:       150
Systems seen by PADLOC only:   144


## Section 3 — Merge DF + PADLOC → defence feature block

### Strategy

Both long-form tables share the same canonical name vocabulary (resolved in Sections 1–2). The merge proceeds in three stages:

**Stage A — count instances per (genome, system, tool).**  
Tag each long-form table with its tool name, concatenate, then groupby (genome_id, canonical_name, tool) and count rows. This gives DF and PADLOC instance counts side by side.

**Stage B — compute presence, count, n_tools per (genome, system).**  
- `presence` = 1 if either tool's count > 0 (union)  
- `count` = max(df_count, padloc_count) — takes the higher estimate without forcing a primary tool  
- `n_tools` = how many tools detected it (0, 1, or 2); used as a quality/confidence column, not a primary ML feature  

**Stage C — pivot to wide, reindex, sparsity filter.**  
Pivot `presence` to get a genome × system binary matrix. Reindex against the full 900-genome list (fills absent genomes with 0). Drop any system column present in <5 genomes across all 900 — these are too rare to be informative and add noise to distance metrics.

### Anti-defence: simpler

PADLOC does not predict anti-defence systems. `df_antidefense_long` goes straight to a pivot — no merge needed. Same sparsity filter applied.

In [11]:
# ── Stage A: count instances per (genome, system, tool) ─────────────────────
# Tag each table with its source tool before concatenating
combined_long = pd.concat([
    df_defense_long.assign(tool="df"),
    padloc_long.assign(tool="padloc"),
], ignore_index=True)

# groupby gives us: for each (genome, system, tool), how many instances?
tool_counts = (
    combined_long
    .groupby(["genome_id", "canonical_name", "tool"])
    .size()                             # row count = instance count
    .reset_index(name="count")
)

# Pivot tool axis: index = (genome_id, canonical_name), columns = tool names
counts_by_tool = tool_counts.pivot_table(
    index=["genome_id", "canonical_name"],
    columns="tool",
    values="count",
    fill_value=0,       # 0 means that tool detected nothing for this (genome, system)
).reset_index()

# Rename columns for clarity (pivot creates a MultiIndex-like columns object)
counts_by_tool.columns.name = None
for col in ["df", "padloc"]:       # ensure both columns exist even if one tool had no hits
    if col not in counts_by_tool.columns:
        counts_by_tool[col] = 0

print(f"(genome, system) pairs detected by at least one tool: {len(counts_by_tool):,}")
print(f"Preview:")
print(counts_by_tool.head(6).to_string())

(genome, system) pairs detected by at least one tool: 12,474
Preview:
         genome_id  canonical_name   df  padloc
0  GCF_000025565.1         PD-T4-6  0.0     1.0
1  GCF_000025565.1            VSPR  0.0     1.0
2  GCF_000025565.1         df_AbiJ  1.0     0.0
3  GCF_000025565.1       df_CapRel  1.0     0.0
4  GCF_000025565.1  df_Mok_Hok_Sok  1.0     0.0
5  GCF_000025565.1      df_PD-T7-1  1.0     0.0


In [12]:
# ── Stage B: compute presence, count, n_tools ────────────────────────────────
counts_by_tool["presence"] = 1   # every row here is a detected (genome, system) pair
counts_by_tool["count"]    = counts_by_tool[["df", "padloc"]].max(axis=1)
counts_by_tool["n_tools"]  = (
    (counts_by_tool["df"] > 0).astype(int) +
    (counts_by_tool["padloc"] > 0).astype(int)
)

print("n_tools distribution (how many tools agreed on each (genome, system) detection):")
print(counts_by_tool["n_tools"].value_counts().sort_index().to_string())
print()

# Spot-check: key systems from the published paper
for sys in ["SspBCDE", "Gao_Qat", "RM_Type_I", "AbiE"]:
    subset = counts_by_tool[counts_by_tool["canonical_name"] == sys]
    n2 = (subset["n_tools"] == 2).sum()
    n1 = (subset["n_tools"] == 1).sum()
    total = len(subset)
    print(f"  {sys:20s}: {total} genomes | both tools={n2} | one tool={n1}")

n_tools distribution (how many tools agreed on each (genome, system) detection):
n_tools
1    8874
2    3600

  SspBCDE             : 82 genomes | both tools=81 | one tool=1
  Gao_Qat             : 75 genomes | both tools=74 | one tool=1
  RM_Type_I           : 581 genomes | both tools=402 | one tool=179
  AbiE                : 305 genomes | both tools=211 | one tool=94


In [13]:
# ── Stage C: pivot to wide, reindex, sparsity filter ─────────────────────────

# Pivot presence → genome × system binary matrix
defence_pa = (
    counts_by_tool
    .pivot(index="genome_id", columns="canonical_name", values="presence")
    .fillna(0)          # genomes absent from this system's rows → 0
    .astype(int)
)

# Pivot count → genome × system count matrix
defence_count = (
    counts_by_tool
    .pivot(index="genome_id", columns="canonical_name", values="count")
    .fillna(0)
    .astype(int)
)

# Reindex both against the full 900-genome list
# Genomes that had ZERO systems in both tools get all-zero rows
defence_pa    = defence_pa.reindex(all_genome_ids, fill_value=0)
defence_count = defence_count.reindex(all_genome_ids, fill_value=0)

print(f"Before sparsity filter: {defence_pa.shape[1]} defence systems")

# Sparsity filter: drop systems present in < 5 genomes
system_prevalence = defence_pa.sum(axis=0)   # number of genomes with presence=1
keep_systems = system_prevalence[system_prevalence >= 5].index

defence_pa    = defence_pa[keep_systems]
defence_count = defence_count[keep_systems]

print(f"After sparsity filter (>=5 genomes): {defence_pa.shape[1]} defence systems")
print(f"Systems dropped (too rare): {len(system_prevalence) - len(keep_systems)}")
print(f"Defence P/A matrix shape: {defence_pa.shape}")
print()
print("Sparsity of final defence P/A matrix:")
print(f"  {(defence_pa == 0).values.mean():.1%}")

Before sparsity filter: 407 defence systems
After sparsity filter (>=5 genomes): 274 defence systems
Systems dropped (too rare): 133
Defence P/A matrix shape: (900, 274)

Sparsity of final defence P/A matrix:
  95.1%


In [14]:
# ── Anti-defence: straight pivot (DF only, no merge needed) ─────────────────
adef_pa = (
    df_antidefense_long
    .groupby(["genome_id", "canonical_name"])
    .size()
    .unstack(fill_value=0)
    .clip(upper=1)                       # presence/absence
    .reindex(all_genome_ids, fill_value=0)
)

# Sparsity filter — same threshold
adef_prevalence = adef_pa.sum(axis=0)
keep_adef = adef_prevalence[adef_prevalence >= 5].index
adef_pa = adef_pa[keep_adef]

print(f"Anti-defence P/A matrix: {adef_pa.shape}  (after sparsity filter)")
print(f"Anti-defence systems dropped: {len(adef_prevalence) - len(keep_adef)}")
print()

# ── Summary of defence feature block ─────────────────────────────────────────
print("=" * 55)
print("DEFENCE FEATURE BLOCK SUMMARY")
print("=" * 55)
print(f"Genomes (rows)         : {defence_pa.shape[0]}")
print(f"Defence systems (cols) : {defence_pa.shape[1]}")
print(f"Anti-defence (cols)    : {adef_pa.shape[1]}")
print(f"Total feature columns  : {defence_pa.shape[1] + adef_pa.shape[1]}")
print()
# Verify key systems from published paper survived the filter
for sys in ["SspBCDE", "Gao_Qat", "RM_Type_I", "RM_Type_II", "RM_Type_IV"]:
    status = "PRESENT" if sys in defence_pa.columns else "DROPPED"
    if status == "PRESENT":
        prev = int(defence_pa[sys].sum())
        print(f"  {sys:20s}: {status} ({prev}/900 genomes)")
    else:
        print(f"  {sys:20s}: {status}")

Anti-defence P/A matrix: (900, 29)  (after sparsity filter)
Anti-defence systems dropped: 11

DEFENCE FEATURE BLOCK SUMMARY
Genomes (rows)         : 900
Defence systems (cols) : 274
Anti-defence (cols)    : 29
Total feature columns  : 303

  SspBCDE             : PRESENT (82/900 genomes)
  Gao_Qat             : PRESENT (75/900 genomes)
  RM_Type_I           : PRESENT (581/900 genomes)
  RM_Type_II          : PRESENT (249/900 genomes)
  RM_Type_IV          : PRESENT (391/900 genomes)


## Section 4 — Parse ResFinder outputs (ARG counts)

### Why ARG count is the Q2 target, not just another feature

The published *Acinetobacter* paper found that RM systems negatively correlate with ARG count — RM acts as a restriction barrier against incoming MGEs that carry ARGs. Q2 asks whether this generalises: can the defence system profile predict high-ARG-burden genomes across ESKAPE?

To test this, we need ARG count as a *label* (the thing we're predicting) and as a *feature* (correlation context). Both go in the matrix; the Q2 target variable is built from the ARG count column in Section 9.

### What counts as one ARG

ResFinder outputs one row per HMM hit — if `blaTEM-1` appears on three plasmids, there are three rows. We compute two ARG metrics:

- `arg_count_unique`: distinct gene names per genome — "how many different resistance genes does this genome carry?" This is the primary burden metric and the Q2 label basis.
- `arg_count_total`: total hits per genome — includes multi-copy genes, a proxy for MGE load.

Both are reported. For Q2, we use `arg_count_unique`.

### File location

`data/interim/{species}/resfinder/{accession}/ResFinder_results_tab.txt` — tab-separated, header row present.

In [15]:
arg_records = []   # one dict per genome

for sp in SPECIES:
    res_dir = INTERIM / sp / "resfinder"   # each genome has its own subdirectory

    for genome_dir in sorted(res_dir.iterdir()):
        if not genome_dir.is_dir():
            continue  # skip .DS_Store

        results_file = genome_dir / "ResFinder_results_tab.txt"

        if not results_file.exists():
            # ResFinder was called but produced no output — treat as zero ARGs
            arg_records.append({
                "genome_id": norm_acc(genome_dir.name),
                "species": sp,
                "arg_count_unique": 0,
                "arg_count_total": 0,
            })
            continue

        raw = pd.read_csv(results_file, sep="\t")

        # ResFinder header: Resistance gene | Identity | ... | Phenotype | ...
        # Column 0 is the gene name — strip it from 'Resistance gene' header
        # Some versions use slightly different headers; guard with fallback
        if "Resistance gene" in raw.columns:
            gene_col = "Resistance gene"
        else:
            gene_col = raw.columns[0]

        genome_id = norm_acc(genome_dir.name)

        if raw.empty:
            arg_records.append({
                "genome_id": genome_id, "species": sp,
                "arg_count_unique": 0, "arg_count_total": 0,
            })
            continue

        arg_records.append({
            "genome_id": genome_id,
            "species":   sp,
            "arg_count_unique": raw[gene_col].nunique(),  # distinct gene names
            "arg_count_total":  len(raw),                  # all hits including duplicates
        })

arg_df = pd.DataFrame(arg_records).set_index("genome_id")

print(f"ARG records parsed: {len(arg_df):,}  (expect 900)")
print(f"Genomes with >=1 ARG: {(arg_df['arg_count_unique'] > 0).sum()}")
print(f"Genomes with zero ARGs: {(arg_df['arg_count_unique'] == 0).sum()}")
print()
print("arg_count_unique distribution:")
print(arg_df["arg_count_unique"].describe().round(1).to_string())
print()
print("Per-species median ARG count (unique):")
print(arg_df.groupby("species")["arg_count_unique"].median().sort_values(ascending=False).to_string())

ARG records parsed: 900  (expect 900)
Genomes with >=1 ARG: 884
Genomes with zero ARGs: 16

arg_count_unique distribution:
count    900.0
mean       8.6
std        6.1
min        0.0
25%        4.0
50%        7.0
75%       12.0
max       34.0

Per-species median ARG count (unique):
species
kpneumoniae    15.5
abaumannii     10.0
efaecium        8.0
paeruginosa     6.0
ecloaceae       5.5
saureus         3.0


## Section 5 — Parse ICEberg outputs (IME/ICE counts)

### Database structure and the coverage ambiguity

The ICEberg tBLASTn database contains protein sequences from known Integrative and Mobilizable Elements (IMEs). Multiple proteins from the same element share the same FASTA header prefix (and therefore the same BLAST `qseqid`). For example, all 5 proteins from element `ICEberg|10_IME` all produce hits with `qseqid = ICEberg|10_IME`.

This means `qstart`/`qend` from a BLAST hit are amino acid coordinates within *one* of those proteins, but the output does not record *which*. To apply the pre-registered 80% query coverage filter, we:
1. Parse the ICEberg FASTA to find the **maximum protein length** per element ID.
2. Use that maximum as the coverage denominator: `coverage = (qend - qstart + 1) / max_protein_length`.

This *underestimates* coverage — a short-protein hit will appear to have lower coverage than it actually does against its actual protein. This is the conservative direction: some genuine hits may be filtered, but false positives are suppressed.

### Counting logic

Consistent with the design decision: `ime_count_unique` = number of distinct `qseqid` values (element IDs) that passed the filter per genome. This counts how many different ICE/IME elements are represented. `ime_count_total` = total filtered hits.

In [16]:
from collections import defaultdict

def parse_fasta_max_lengths(fasta_path: Path) -> dict:
    """
    Parse a protein FASTA and return {first_word_of_header: max_sequence_length_aa}.
    When multiple sequences share the same first header word (as in ICEberg),
    the maximum length is kept — used as the conservative coverage denominator.
    """
    lengths = defaultdict(list)
    current_id = None
    current_len = 0
    with open(fasta_path) as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            if line.startswith(">"):
                if current_id is not None:
                    lengths[current_id].append(current_len)
                current_id = line[1:].split()[0]   # first word after '>'
                current_len = 0
            else:
                current_len += len(line)           # amino acid characters
    if current_id is not None:
        lengths[current_id].append(current_len)
    return {k: max(v) for k, v in lengths.items()}


# ── Build ICEberg protein length lookup ──────────────────────────────────────
ICEBERG_FASTA = ROOT / "data" / "raw" / "databases" / "ICEberg_IME.fasta"
ice_lengths = parse_fasta_max_lengths(ICEBERG_FASTA)

print(f"ICEberg unique element IDs with protein lengths: {len(ice_lengths)}")
# Spot-check: element 160 appears in the sample BLAST output
for eid in ["ICEberg|160_IME", "ICEberg|10_IME", "ICEberg|1_ICE"]:
    print(f"  {eid}: max protein length = {ice_lengths.get(eid, 'NOT FOUND')} aa")

ICEberg unique element IDs with protein lengths: 98
  ICEberg|160_IME: max protein length = 1072 aa
  ICEberg|10_IME: max protein length = 559 aa
  ICEberg|1_ICE: max protein length = NOT FOUND aa


In [17]:
BLAST_COLS = ["qseqid", "sseqid", "pident", "length", "mismatch",
              "gapopen", "qstart", "qend", "sstart", "send", "evalue", "bitscore"]

PIDENT_MIN    = 40.0   # % identity threshold
COVERAGE_MIN  = 0.80   # 80% query coverage

ime_records = []

for sp in SPECIES:
    ice_dir = INTERIM / sp / "iceberg"

    for tsv_file in sorted(ice_dir.iterdir()):
        if not tsv_file.name.endswith("_iceberg.tsv"):
            continue

        stem      = tsv_file.stem.removesuffix("_iceberg")
        genome_id = norm_acc(stem)

        # Empty file = zero ICEberg hits (legitimate result)
        if tsv_file.stat().st_size == 0:
            ime_records.append({
                "genome_id": genome_id, "species": sp,
                "ime_count_unique": 0, "ime_count_total": 0,
            })
            continue

        raw = pd.read_csv(tsv_file, sep="\t", header=None, names=BLAST_COLS)

        # ── Apply filters ────────────────────────────────────────────────────
        # 1. pident >= 40%
        raw = raw[raw["pident"] >= PIDENT_MIN]

        # 2. Query coverage >= 80%, using max protein length as denominator
        #    coverage = (qend - qstart + 1) / max_protein_length
        #    Hits with unknown qseqid (not in our length dict) are dropped.
        raw = raw[raw["qseqid"].isin(ice_lengths)]
        raw = raw.copy()
        raw["max_prot_len"] = raw["qseqid"].map(ice_lengths)
        raw["coverage"]     = (raw["qend"] - raw["qstart"] + 1) / raw["max_prot_len"]
        raw = raw[raw["coverage"] >= COVERAGE_MIN]

        ime_records.append({
            "genome_id":       genome_id,
            "species":         sp,
            "ime_count_unique": raw["qseqid"].nunique(),  # distinct element IDs
            "ime_count_total":  len(raw),                  # all passing hits
        })

ime_df = pd.DataFrame(ime_records).set_index("genome_id")

print(f"IME records: {len(ime_df):,}  (expect 900)")
print(f"Genomes with >=1 IME hit: {(ime_df['ime_count_unique'] > 0).sum()}")
print(f"Genomes with zero IMEs:   {(ime_df['ime_count_unique'] == 0).sum()}")
print()
print("ime_count_unique distribution:")
print(ime_df["ime_count_unique"].describe().round(1).to_string())
print()
print("Per-species median IME count (unique):")
print(ime_df.groupby("species")["ime_count_unique"].median().sort_values(ascending=False).to_string())

IME records: 900  (expect 900)
Genomes with >=1 IME hit: 805
Genomes with zero IMEs:   95

ime_count_unique distribution:
count    900.0
mean       9.9
std        6.9
min        0.0
25%        3.8
50%       11.0
75%       15.0
max       46.0

Per-species median IME count (unique):
species
kpneumoniae    17.0
ecloaceae      14.5
efaecium       12.0
abaumannii     11.0
saureus         4.0
paeruginosa     2.0


## Section 6 — Parse BacMet outputs (HMRG counts) [SUPERSEDED — see Section 6b]

BacMet (Heavy Metal Resistance Gene database) records resistance genes for metals: arsenic, copper, mercury, zinc, cobalt, etc. Heavy metal resistance genes are co-selected with ARGs on the same MGEs — metal contamination in the environment selects for resistance cassettes that carry both.

The BLAST structure is identical to ICEberg, but BacMet has **unique qseqids per protein** (`BAC0001|abeM|...`), so query coverage is exact: `coverage = (qend - qstart + 1) / protein_length`.

Same BLAST filters: pident ≥ 40%, coverage ≥ 80%.

Counting: `hmrg_count_unique` = distinct BacMet protein IDs passing filters. `hmrg_count_total` = all passing hits.

**This section is retained as a historical record only.** `hmrg_df` produced below is NOT included in the Section 9 join. BacMet was removed due to gram-stain reference bias — see `docs/decisions.md` (2026-05-12) and Section 6b for the AMRFinderPlus replacement.

In [18]:
BACMET_FASTA = ROOT / "data" / "raw" / "databases" / "BacMet2_EXP_database.fasta"

# BacMet: each protein has a unique qseqid → exact coverage calculation
# parse_fasta_max_lengths works here too: max of a single value = the value itself
bacmet_lengths = parse_fasta_max_lengths(BACMET_FASTA)

print(f"BacMet unique protein IDs: {len(bacmet_lengths)}")
for pid in ["BAC0001|abeM|tr|Q5FAM9|Q5FAM9_ACIBA",
            "BAC0002|abeS|tr|Q2FD83|Q2FD83_ACIBA",
            "BAC0005|acrA|sp|P0AE06|ACRA_ECOLI"]:
    print(f"  {pid[:45]}: {bacmet_lengths.get(pid, 'NOT FOUND')} aa")

print()

hmrg_records = []

for sp in SPECIES:
    bm_dir = INTERIM / sp / "bacmet"

    for tsv_file in sorted(bm_dir.iterdir()):
        if not tsv_file.name.endswith("_bacmet.tsv"):
            continue

        stem      = tsv_file.stem.removesuffix("_bacmet")
        genome_id = norm_acc(stem)

        if tsv_file.stat().st_size == 0:
            hmrg_records.append({
                "genome_id": genome_id, "species": sp,
                "hmrg_count_unique": 0, "hmrg_count_total": 0,
            })
            continue

        raw = pd.read_csv(tsv_file, sep="\t", header=None, names=BLAST_COLS)

        # pident filter
        raw = raw[raw["pident"] >= PIDENT_MIN]

        # Coverage filter — exact for BacMet (one protein per qseqid)
        raw = raw[raw["qseqid"].isin(bacmet_lengths)]
        raw = raw.copy()
        raw["prot_len"]  = raw["qseqid"].map(bacmet_lengths)
        raw["coverage"]  = (raw["qend"] - raw["qstart"] + 1) / raw["prot_len"]
        raw = raw[raw["coverage"] >= COVERAGE_MIN]

        hmrg_records.append({
            "genome_id":        genome_id,
            "species":          sp,
            "hmrg_count_unique": raw["qseqid"].nunique(),
            "hmrg_count_total":  len(raw),
        })

hmrg_df = pd.DataFrame(hmrg_records).set_index("genome_id")

print(f"HMRG records: {len(hmrg_df):,}  (expect 900)")
print(f"Genomes with >=1 HMRG: {(hmrg_df['hmrg_count_unique'] > 0).sum()}")
print(f"Genomes with zero HMRGs: {(hmrg_df['hmrg_count_unique'] == 0).sum()}")
print()
print("hmrg_count_unique distribution:")
print(hmrg_df["hmrg_count_unique"].describe().round(1).to_string())
print()
print("Per-species median HMRG count (unique):")
print(hmrg_df.groupby("species")["hmrg_count_unique"].median().sort_values(ascending=False).to_string())

BacMet unique protein IDs: 753
  BAC0001|abeM|tr|Q5FAM9|Q5FAM9_ACIBA: 448 aa
  BAC0002|abeS|tr|Q2FD83|Q2FD83_ACIBA: 109 aa
  BAC0005|acrA|sp|P0AE06|ACRA_ECOLI: 397 aa



HMRG records: 900  (expect 900)
Genomes with >=1 HMRG: 900
Genomes with zero HMRGs: 0

hmrg_count_unique distribution:
count    900.0
mean     203.4
std      114.3
min       39.0
25%       62.0
50%      221.0
75%      305.0
max      390.0

Per-species median HMRG count (unique):
species
kpneumoniae    330.0
ecloaceae      307.5
paeruginosa    282.5
abaumannii     177.0
saureus         61.0
efaecium        55.0


### BacMet caveat — high counts expected, interpretation is conservative

All 900 genomes have ≥1 HMRG hit; median is 221 distinct BacMet proteins per genome. This is not a pipeline error.

The BacMet EXP database includes many **chromosomally encoded housekeeping proteins** with metal-handling functions (RND efflux pumps, copper-transporting ATPases, arsenic detoxification enzymes) that are broadly conserved across bacteria. At pident ≥ 40%, distant homologs match in virtually every genome.

Implications:
- `hmrg_count_unique` does not cleanly separate "has acquired heavy metal resistance" from "has normal bacterial metal homeostasis machinery." It reflects both.
- This makes HMRG count a weaker proxy for MGE-acquired metal resistance than ARG count is for antibiotic resistance, because ResFinder's curated acquired-gene focus is stricter than BacMet's experimentally verified proteins.
- For ML features: `hmrg_count_unique` will likely carry species-level information (gram-positive vs gram-negative baseline metal homeostasis genes differ) more than MGE-load information. Flag in the Methods/Discussion.
- The correlation with ARG/IME counts (the paper's main question) is still testable — it just should be interpreted as "co-occurrence of metal + antibiotic resistance gene content" not as "co-acquisition on the same MGE."

## Section 6b — AMRFinderPlus metal resistance genes

### Why this replaces BacMet

The BacMet tBLASTn run (Section 6 above) captured constitutive RND efflux pump structural homologs — gram-negative housekeeping proteins — not acquired metal resistance genes. At any pident threshold, KP genomes have ~6× more BacMet hits than SA genomes, and the ratio grows to ~28× at pident ≥ 80% as the gram-negative-dominated reference proteins become more discriminating. This is a reference composition artefact, not a biological signal.

AMRFinderPlus with `--plus --organism` resolves both problems:

**`--plus`** unlocks the STRESS category. Without this flag, AMRFinderPlus reports only core AMR genes (β-lactamases, aminoglycosides, etc.). The METAL subtype lives under STRESS → METAL, and is completely absent from the output without `--plus`.

**`--organism`** loads species-specific HMM profiles that exist outside the pan-bacterial core database. SA copper resistance genes (CopA, CopZ, Mco) and EF cadmium resistance genes (cadD, cadA) are defined only in organism-specific HMM sets. Without `--organism Staphylococcus_aureus`, AMRFinderPlus searches only the pan-bacterial core — and these genes simply do not appear there. They are not "missed at low confidence" — they are not in the search at all.

### Filter: `Subtype == "METAL"`

AMRFinderPlus output structure: `Type` (e.g. STRESS, AMR, VIRULENCE) → `Subtype` (METAL, BIOCIDE, ACID, POINT, ...) → `Class` (MERCURY, ARSENIC, COPPER, ...).

We filter `Subtype == "METAL"` to extract only metal resistance gene annotations.

### Design decision: actual Class values differ from the 6-class plan

| Class | Hits | Biological locus |
|---|---|---|
| MERCURY | 1684 | *mer* operon (merA, merB, merC, merD, merT) |
| COPPER/SILVER | 1349 | *pco/sil* gene cluster — co-resistance locus on IncH plasmids |
| COPPER | 1231 | *cup* / *cop* operon — standalone CopA/CopB |
| ARSENIC | 1092 | *ars* operon (arsA, arsB, arsC, arsR) |
| TELLURIUM | 358 | *ter* operon (terA–terF) |
| SILVER | 354 | *sil* operon standalone |
| NA | 348 | Detected but no Class assigned (e.g. fieF, generic cation efflux pump) |
| NICKEL | 230 | *nik/rcn* operons — not in original plan |
| CADMIUM | 56 | *cad* operon (SA/EF-specific) |
| COPPER/NICKEL | 34 | *czc* operon variants |
| CADMIUM/LEAD/ZINC | 5 | *czc/cad* variants |
| CHROMATE | 4 | *chr* operon |

No standalone ZINC class — zinc resistance is embedded in compound annotations only. Compound classes (COPPER/SILVER) are real genetic loci, not annotation ambiguity: the *pco/sil* cluster co-encodes copper and silver resistance on a single operon.

### Caveat: 82 crash genomes coded as zero

`amr_report` v4.2.7 crashes on specific protein sequences (confirmed bug). A 3-tier fallback was run (organism → pan-bacterial → header-only TSV). The fallback recovered zero — all 82 crash regardless of `--organism`. Affected: SA 53/150 (35%), EF 28/150 (19%), AB 1/150. Coded as zero metal resistance genes in the feature matrix. SA and EF metal resistance analyses must include a Methods caveat.

In [19]:
# ── Metal class values observed in the actual AMRFinderPlus data ──────────────
# Original plan listed 6 classes; actual data has 12 (see markdown above).
METAL_CLASSES = [
    "MERCURY", "ARSENIC",
    "COPPER", "SILVER", "COPPER/SILVER",      # pco/sil variants
    "TELLURIUM",
    "NICKEL", "CADMIUM",
    "COPPER/NICKEL", "CADMIUM/LEAD/ZINC", "CHROMATE",
]

def metal_col(cls: str) -> str:
    """COPPER/SILVER → hmrg_copper_silver  (lowercase, / → _)"""
    return "hmrg_" + cls.lower().replace("/", "_")

amr_records = []

for sp in SPECIES:
    amr_dir = INTERIM / sp / "amrfinderplus"

    for tsv_file in sorted(amr_dir.iterdir()):
        if not tsv_file.name.endswith("_amrfinder.tsv"):
            continue

        stem      = tsv_file.stem.removesuffix("_amrfinder")
        genome_id = norm_acc(stem)

        record = {"genome_id": genome_id, "species": sp}
        for cls in METAL_CLASSES:
            record[metal_col(cls)] = 0
        record["hmrg_unclassified"] = 0   # Class == "NA" rows
        record["hmrg_metal_total"]  = 0
        record["hmrg_metal_classes"] = 0

        # Header-only file = amr_report crash fallback (82 genomes) — treat as zero
        if tsv_file.stat().st_size == 0:
            amr_records.append(record)
            continue

        try:
            df_amr = pd.read_csv(tsv_file, sep="\t")
        except Exception:
            amr_records.append(record)
            continue

        if "Subtype" not in df_amr.columns or df_amr.empty:
            # 0-row DataFrame = header-only crash fallback
            amr_records.append(record)
            continue

        metal = df_amr[df_amr["Subtype"] == "METAL"].copy()

        if metal.empty:
            amr_records.append(record)
            continue

        record["hmrg_metal_total"] = len(metal)

        class_counts = metal["Class"].value_counts()
        for cls in METAL_CLASSES:
            record[metal_col(cls)] = int(class_counts.get(cls, 0))
        record["hmrg_unclassified"] = int(class_counts.get("NA", 0))

        # Metal class diversity: distinct Class values, NA excluded
        record["hmrg_metal_classes"] = int(
            metal[metal["Class"] != "NA"]["Class"].nunique()
        )

        amr_records.append(record)

amrfinder_df = pd.DataFrame(amr_records).set_index("genome_id")

print(f"AMRFinderPlus records: {len(amrfinder_df):,}  (expect 900)")
print(f"Genomes with >=1 METAL gene:              {(amrfinder_df['hmrg_metal_total'] > 0).sum()}")
print(f"Genomes coded as zero (crash or genuine): {(amrfinder_df['hmrg_metal_total'] == 0).sum()}")
print()
print("hmrg_metal_total distribution:")
print(amrfinder_df["hmrg_metal_total"].describe().round(1).to_string())
print()
print("Per-species median metal gene count:")
print(amrfinder_df.groupby("species")["hmrg_metal_total"].median()
      .sort_values(ascending=False).to_string())
print()
print("Per-species median metal class diversity (hmrg_metal_classes):")
print(amrfinder_df.groupby("species")["hmrg_metal_classes"].median()
      .sort_values(ascending=False).to_string())
print()
print("Metal class distribution (total hits across all 900 genomes):")
metal_col_list = [metal_col(cls) for cls in METAL_CLASSES]
print(amrfinder_df[metal_col_list].sum().sort_values(ascending=False).to_string())

AMRFinderPlus records: 900  (expect 900)
Genomes with >=1 METAL gene:              753
Genomes coded as zero (crash or genuine): 147

hmrg_metal_total distribution:
count    900.0
mean       7.5
std       11.3
min        0.0
25%        1.0
50%        2.0
75%        9.0
max       66.0

Per-species median metal gene count:
species
kpneumoniae    20.0
ecloaceae      19.0
paeruginosa     2.0
abaumannii      1.0
efaecium        1.0
saureus         1.0

Per-species median metal class diversity (hmrg_metal_classes):
species
ecloaceae      4.0
kpneumoniae    4.0
abaumannii     1.0
efaecium       1.0
paeruginosa    1.0
saureus        1.0

Metal class distribution (total hits across all 900 genomes):
hmrg_mercury              1684
hmrg_copper_silver        1349
hmrg_copper               1231
hmrg_arsenic              1092
hmrg_tellurium             358
hmrg_silver                354
hmrg_nickel                230
hmrg_cadmium                56
hmrg_copper_nickel          34
hmrg_cadmium_lead_zin

## Section 7 — ISEScan (IS element counts by family)

### Why IS elements are a separate feature from ICE/IME counts

Section 5 counted ICEs and IMEs — composite mobile elements carrying dozens of genes including conjugation machinery. A single ICE typically appears 1–3 times per genome and contributes ~15–30 BLAST hits from all its genes.

IS elements are structurally different: a single ORF encoding the transposase, flanked by inverted terminal repeats (IRs), totalling 750–2500 bp. They move independently, without conjugation functions. They are also far more abundant — a single clinical *K. pneumoniae* genome can carry 90+ IS elements from multiple families.

**Why IS family identity matters:**
- **IS6/IS26**: flanks ARG-bearing composite transposons in clinical isolates. High IS26 copy number in KP is a known marker of extensive resistance cassette accumulation.
- **IS3**: preferentially inserts near promoter-proximal sites; can activate adjacent resistance genes by providing an outward-facing promoter in its IR sequence.
- **ISL3**: dominant in gram-positive organisms (high prevalence in EF). Associated with resistance gene mobilisation in enterococci.
- **`new` family**: elements with complete IR + transposase structure that ISEScan could not assign to a known family. These are real IS elements with all the biology described above — they are unclassified by database coverage, not artefacts. Included as `is_new_count`.

### File structure

`data/interim/{sp}/isescan/{acc_fn}_isescan.tsv` — one row per IS element call, columns: `seqID`, `family`, `cluster`, `isBegin`, `isEnd`, `isLen`, `type` (c = complete, p = partial), and others.

32 empty files across 900 genomes = legitimate zero-IS assemblies (most are SA or PA).

### Output

`is_df`: index = genome_id, columns = `is_count_total` + `is_{family}_count` for each of the 25 IS families observed. No sparsity filter here — Section 9 handles it globally.

In [20]:
is_record_list = []

for sp in SPECIES:
    is_dir = INTERIM / sp / "isescan"

    for tsv_file in sorted(is_dir.iterdir()):
        if not tsv_file.name.endswith("_isescan.tsv"):
            continue

        stem      = tsv_file.stem.removesuffix("_isescan")
        genome_id = norm_acc(stem)

        # Empty file = zero IS elements (~32 genomes — legitimate)
        if tsv_file.stat().st_size == 0:
            is_record_list.append({
                "genome_id": genome_id, "species": sp, "is_count_total": 0
            })
            continue

        try:
            df_is = pd.read_csv(tsv_file, sep="\t")
        except Exception:
            is_record_list.append({
                "genome_id": genome_id, "species": sp, "is_count_total": 0
            })
            continue

        if df_is.empty or "family" not in df_is.columns:
            is_record_list.append({
                "genome_id": genome_id, "species": sp, "is_count_total": 0
            })
            continue

        # Count IS elements per family for this genome
        family_counts = df_is["family"].value_counts()
        record = {
            "genome_id":      genome_id,
            "species":        sp,
            "is_count_total": len(df_is),   # all IS elements (complete + partial)
        }
        for fam, cnt in family_counts.items():
            record[f"is_{fam}_count"] = int(cnt)   # e.g. is_IS3_count, is_new_count

        is_record_list.append(record)

# fillna(0): families absent from a given genome get count 0
is_df = (
    pd.DataFrame(is_record_list)
    .fillna(0)
    .set_index("genome_id")
)

# Convert all IS count columns to int (fillna produces float)
is_count_cols = [c for c in is_df.columns if c.startswith("is_") and c.endswith("_count")]
is_df[is_count_cols + ["is_count_total"]] = (
    is_df[is_count_cols + ["is_count_total"]].astype(int)
)

is_families = sorted(c for c in is_df.columns if c.startswith("is_") and c != "is_count_total")

print(f"IS element records: {len(is_df):,}  (expect 900)")
print(f"Genomes with >=1 IS element:   {(is_df['is_count_total'] > 0).sum()}")
print(f"Genomes with zero IS elements: {(is_df['is_count_total'] == 0).sum()}  (expect ~32)")
print(f"IS families tracked: {len(is_families)}")
print()
print("is_count_total distribution:")
print(is_df["is_count_total"].describe().round(1).to_string())
print()
print("Per-species median IS element count:")
print(is_df.groupby("species")["is_count_total"].median()
      .sort_values(ascending=False).to_string())
print()
print("Top 10 IS families by total element count across all 900 genomes:")
family_totals = is_df[is_families].sum().sort_values(ascending=False)
print(family_totals.head(10).to_string())

IS element records: 900  (expect 900)
Genomes with >=1 IS element:   868
Genomes with zero IS elements: 32  (expect ~32)
IS families tracked: 25

is_count_total distribution:
count    900.0
mean      56.1
std       52.4
min        0.0
25%       19.0
50%       40.0
75%       69.0
max      261.0

Per-species median IS element count:
species
efaecium       154.0
kpneumoniae     61.0
ecloaceae       42.0
abaumannii      39.5
saureus         22.0
paeruginosa     18.0

Top 10 IS families by total element count across all 900 genomes:
is_IS3_count       8764
is_ISL3_count      6327
is_IS6_count       4647
is_IS5_count       4451
is_IS30_count      4078
is_IS256_count     3936
is_IS4_count       2588
is_IS110_count     2291
is_IS21_count      2102
is_IS1182_count    1817


### Sections 6b and 7 comprehension check

Answer before moving to Section 8.

**Q1 — AMRFinderPlus `--organism` mechanism (revisit)**

When you run `amrfinder --protein genome.prt --plus` on a *S. aureus* genome, SA copper resistance genes (CopA/CopZ) produce zero hits. Add `--organism Staphylococcus_aureus` and they appear. What specifically does `--organism` change about the tool's search that explains this result?

**Q2 — COPPER/SILVER compound class**

The data shows COPPER/SILVER (1349 hits) is more frequent than COPPER alone (1231) or SILVER alone (354). Why does AMRFinderPlus report copper and silver resistance as a single compound Class rather than two separate entries? What does this tell you about the underlying genetic locus these genes come from?

**Q3 — `new` family in IS data**

You are about to use IS family counts as ML features. A reviewer questions including the `new` family (ISEScan's label for IS elements it identified structurally but could not classify into a known family). Write a 2-sentence rebuttal defending their inclusion.

**Q4 — IS count vs IME count interpretation**

Genome A: `ime_count_unique = 5`, `is_count_total = 90`. Genome B: `ime_count_unique = 5`, `is_count_total = 10`. Same ICE/IME burden. What does the IS count difference tell you about these genomes' MGE ecology, and which would you predict has higher ARG burden?

## Section 8 — MLST and metadata

### What MLST gives us and why it belongs in the analysis

MLST types each genome by recording the allele number at 7 slowly-evolving housekeeping gene loci. The combination of 7 allele numbers maps to a Sequence Type (ST). Two genomes sharing an ST are almost certainly clonal — descended from a common ancestor within recent decades.

We are **not** using ST as a predictive feature for Q1 or Q2. It serves two specific purposes:

1. **Phase 9 grouped k-fold CV.** Instead of randomly splitting 900 genomes across CV folds, we group all genomes sharing an ST into the same fold. This prevents the classifier from memorising a lineage-specific defence profile during training and recognising it in test — giving an inflated accuracy that has nothing to do with generalising defence-system rules across clades.
2. **Species identity QC.** Each MLST scheme is species-specific. If a genome downloaded as *K. pneumoniae* types under the *E. coli* scheme, its core housekeeping genes are *E. coli*, not *Klebsiella* — a species misidentification. CheckM2 tests completeness and contamination level but does not verify taxonomic identity; MLST does.

### What metadata gives us

Country and `year_bin` are potential confounders: a dataset skewed toward European KP isolates may embed a geographic defence-profile artefact rather than a biological signal. Recording these allows us to check for geographic bias in Phase 9 and to stratify train/test splits if necessary. For E. cloacae complex, `complex_member` identifies which of the six sub-taxa each genome belongs to — important because the ML treats the complex as one class, but its internal diversity should be reported.

In [21]:
# ── Expected MLST scheme per species ────────────────────────────────────────
# Any genome whose scheme does not match is flagged as a potential species
# misidentification — CheckM2 QC does not catch this, MLST does.
EXPECTED_SCHEME = {
    "abaumannii":  "abaumannii_2",
    "ecloaceae":   "ecloacae",       # E. cloacae complex all use this scheme
    "efaecium":    "efaecium",
    "kpneumoniae": "klebsiella",
    "paeruginosa": "paeruginosa",
    "saureus":     "saureus",
}

mlst_records = []

for sp in SPECIES:
    mlst_file = INTERIM / sp / "mlst" / "mlst_results.tsv"

    # No header; columns: filepath \t scheme \t ST \t allele1 \t allele2 ...
    df_raw = pd.read_csv(
        mlst_file, sep="\t", header=None,
        usecols=[0, 1, 2],
        names=["filepath", "scheme", "st_raw"],
        dtype=str,
    )

    for _, row in df_raw.iterrows():
        genome_id = norm_acc(Path(row["filepath"]).stem)   # stem = GCF_XXX_1, norm_acc → GCF_XXX.1
        scheme    = row["scheme"].strip()
        st_raw    = row["st_raw"].strip()

        # Parse ST: "-" = untypeable; any "?" or "~" in the value = novel ST → NaN
        if st_raw == "-" or "?" in st_raw or "~" in st_raw:
            sequence_type = pd.NA
        else:
            try:
                sequence_type = int(st_raw)
            except ValueError:
                sequence_type = pd.NA

        # scheme "-" means typing failed entirely (genome issue, not species mismatch)
        scheme_mismatch = (scheme != EXPECTED_SCHEME[sp]) and (scheme != "-")

        mlst_records.append({
            "genome_id":       genome_id,
            "species":         sp,
            "mlst_scheme":     scheme,
            "sequence_type":   sequence_type,
            "scheme_mismatch": scheme_mismatch,
        })

mlst_df = (
    pd.DataFrame(mlst_records)
    .set_index("genome_id")
    .astype({"species": "category", "mlst_scheme": "category", "scheme_mismatch": bool})
)

# ── QC report ─────────────────────────────────────────────────────────────
print("── MLST scheme distribution per species ──")
print(mlst_df.groupby(["species", "mlst_scheme"]).size().rename("count").to_string())
print()
n_mismatch = int(mlst_df["scheme_mismatch"].sum())
print(f"Scheme mismatches: {n_mismatch} / {len(mlst_df)}")
print()
print(mlst_df[mlst_df["scheme_mismatch"]][["species", "mlst_scheme", "sequence_type"]])

── MLST scheme distribution per species ──
species      mlst_scheme    
abaumannii   abaumannii_2       150
ecloaceae    -                    1
             cronobacter          4
             ecloacae           145
efaecium     efaecium           150
kpneumoniae  ecoli_achtman_4     18
             klebsiella         132
paeruginosa  paeruginosa        150
saureus      saureus            150

Scheme mismatches: 22 / 900

                     species      mlst_scheme sequence_type
genome_id                                                  
GCF_000239975.1    ecloaceae      cronobacter           431
GCF_003254805.1    ecloaceae      cronobacter           421
GCF_013376815.1    ecloaceae      cronobacter           427
GCF_049600905.1    ecloaceae      cronobacter           340
GCF_003571705.1  kpneumoniae  ecoli_achtman_4         14464
GCF_009661615.2  kpneumoniae  ecoli_achtman_4         14464
GCF_012971225.1  kpneumoniae  ecoli_achtman_4         11307
GCF_013376555.1  kpneumoniae  ecol

### QC finding: 22 genomes carry wrong-species MLST scheme

| Species | Expected | Observed | Count |
|---------|----------|----------|-------|
| kpneumoniae | klebsiella | ecoli_achtman_4 | 18 |
| ecloaceae | ecloacae | cronobacter | 4 |

These 22 genomes passed CheckM2 quality gates (≥95% complete, ≤5% contaminated) but their core housekeeping genes type as a different genus. They are almost certainly species-misidentified NCBI submissions.

**They will be excluded in Section 9** at the final join step. The dataset drops from 900 to 878 genomes. This decision is logged in `docs/decisions.md`.

**Why CheckM2 didn't catch this:** CheckM2 tests completeness using a universal set of single-copy marker genes and measures contamination as the fraction of marker genes present in multiple copies. It does not test whether the marker genes match the expected species. A genome assembled entirely from *E. coli* DNA scores 100% complete and 0% contaminated by CheckM2 — it looks like a perfect *E. coli* genome, which NCBI submitted as *K. pneumoniae*.

In [22]:
# ── Metadata: country, year_bin, complex_member ─────────────────────────────
# metadata.tsv accessions are already in NCBI dot form (GCF_XXXXXXXXX.V).
# No norm_acc conversion needed.

meta_records = []

for sp in SPECIES:
    meta_file = INTERIM / sp / "metadata.tsv"
    df_raw = pd.read_csv(meta_file, sep="\t", dtype=str)

    for _, row in df_raw.iterrows():
        meta_records.append({
            "genome_id":      row["accession"],
            "species":        sp,
            "country":        row.get("country", "unknown"),
            "year_bin":       row.get("year_bin", "unknown"),
            "complex_member": row.get("complex_member", pd.NA),
        })

meta_df = (
    pd.DataFrame(meta_records)
    .set_index("genome_id")
    .astype({"species": "category"})
)

print(f"meta_df: {meta_df.shape[0]} genomes × {meta_df.shape[1]} columns")
print()
print("── Country distribution (top 10 across all species) ──")
print(meta_df["country"].value_counts().head(10).to_string())
print()
print("── Year bin distribution ──")
print(meta_df["year_bin"].value_counts().sort_index().to_string())
print()
print("── E. cloacae complex members ──")
print(meta_df[meta_df["species"] == "ecloaceae"]["complex_member"].value_counts().to_string())

meta_df: 900 genomes × 4 columns

── Country distribution (top 10 across all species) ──
country
unknown           197
China             115
USA               104
South Korea        57
missing            46
India              36
Australia          33
Taiwan             29
Germany            26
United Kingdom     20

── Year bin distribution ──
year_bin
2010-2015        44
2016-2020       403
2021-present    453

── E. cloacae complex members ──
complex_member
E. hormaechei      60
E. cloacae         50
E. asburiae        15
E. kobei           10
E. ludwigii        10
E. roggenkampii     5


### Section 8 comprehension check

Answer before moving to Section 9.

**Q1.** MLST uses housekeeping genes — e.g., for Klebsiella: gapA, infB, mdh, pgi, phoE, rpoB, tonB. Why are these genes chosen for strain typing rather than virulence genes, ARGs, or defence system genes?

**Q2.** Genomes A and B both have `sequence_type = 258` in the KP dataset. Does this mean they came from the same patient? Does it mean they have identical defence system profiles? State precisely what ST-258 tells you and what it does not.

**Q3.** 18 KP genomes passed CheckM2 (≥95% complete, ≤5% contaminated) but MLST types them as *E. coli*. Explain mechanistically — not just "CheckM2 doesn't check species" — why a genome can pass completeness and contamination QC and still be a different species.

## Section 9 — Join all blocks → feature matrix

### What this section does and why each step matters

All seven tool-output dataframes (Sections 1–7) plus MLST and metadata (Section 8) are assembled into one matrix here. Four things happen before the final join:

**1. Exclusion.** 22 genomes with wrong-species MLST scheme are removed. Dataset: 900 → 878.

**2. IS sparsity filter.** IS families present in <5 genomes are dropped (same threshold as defence systems in Section 3). `IS200/IS605` family name contains a `/` — sanitized to `IS200_IS605` for parquet and ML library compatibility.

**3. Derived features.**
- `defence_system_count`: sum of all `dp_*` columns — how many distinct defence types each genome carries. This is the denominator for ratio features.
- `adef_system_count`: sum of all `ad_*` columns.
- `ratio_arg_defence`, `ratio_ime_defence`, `ratio_adef_defence`: count divided by `(defence_system_count + 1)`. The `+1` prevents division by zero for the few genomes with no defence systems at all. Without it, 0/0 → NaN, which propagates silently through ML pipelines.
- `arg_burden_tertile`: Q2 target label. Computed *within each species* using `pd.qcut(q=3)`. Pooled tertiles would make "high_ARG" synonymous with KP and "low_ARG" synonymous with SA — the classifier would learn species identity, not defence-ARG biology.

**4. Column prefixes.** `dp_` = defence P/A (binary primary ML features), `dc_` = defence counts, `ad_` = anti-defence P/A. Prefixes prevent name collisions and let you slice by feature type with `filter(like='dp_')`.

In [23]:
# ── Step 1: identify included genomes ───────────────────────────────────────
mismatch_ids = set(mlst_df.index[mlst_df["scheme_mismatch"]])
included_ids  = [g for g in all_genome_ids if g not in mismatch_ids]

print(f"Excluded (scheme mismatch): {len(mismatch_ids)}")
print(f"Included:                   {len(included_ids)}")
print()
mismatch_by_sp = mlst_df[mlst_df["scheme_mismatch"]].groupby("species")["mlst_scheme"].value_counts()
print("Excluded breakdown:")
print(mismatch_by_sp.to_string())
print()

# ── Step 2: IS element sparsity filter ──────────────────────────────────────
is_family_cols = [c for c in is_df.columns
                  if c.startswith("is_") and c.endswith("_count") and c != "is_count_total"]

# Prevalence = number of genomes (across all 900, pre-exclusion) with count > 0
is_prevalence = (is_df[is_family_cols] > 0).sum(axis=0)
keep_is_cols  = is_prevalence[is_prevalence >= 5].index.tolist()
dropped_is    = [c for c in is_family_cols if c not in keep_is_cols]

print(f"IS families before filter: {len(is_family_cols)}")
print(f"Dropped (<5 genomes):      {len(dropped_is)}")
if dropped_is:
    print(f"  {dropped_is}")
print(f"Retained:                  {len(keep_is_cols)}")
print()

# Sanitise: IS200/IS605 → IS200_IS605 (slash not safe in column names)
is_rename = {c: c.replace("/", "_") for c in keep_is_cols if "/" in c}
if is_rename:
    print(f"Column names sanitised: {is_rename}")

is_df_clean = (
    is_df[["is_count_total"] + keep_is_cols]
    .rename(columns=is_rename)
    .loc[included_ids]
)
print(f"is_df_clean shape: {is_df_clean.shape}")

Excluded (scheme mismatch): 22
Included:                   878

Excluded breakdown:
species      mlst_scheme    
ecloaceae    cronobacter         4
             paeruginosa         0
             klebsiella          0
             efaecium            0
             saureus             0
             abaumannii_2        0
             -                   0
             ecoli_achtman_4     0
             ecloacae            0
kpneumoniae  ecoli_achtman_4    18
             paeruginosa         0
             cronobacter         0
             -                   0
             abaumannii_2        0
             ecloacae            0
             efaecium            0
             klebsiella          0
             saureus             0

IS families before filter: 25
Dropped (<5 genomes):      2
  ['is_ISAS1_count', 'is_IS1634_count']
Retained:                  23

Column names sanitised: {'is_IS200/IS605_count': 'is_IS200_IS605_count'}
is_df_clean shape: (878, 24)


In [24]:
# ── Step 3: filter defence matrices to included genomes ─────────────────────
dp_mat = defence_pa.loc[included_ids]      # presence/absence
dc_mat = defence_count.loc[included_ids]   # copy counts
ad_mat = adef_pa.loc[included_ids]         # anti-defence presence/absence

# ── Summary features ─────────────────────────────────────────────────────────
defence_system_count = dp_mat.sum(axis=1).rename("defence_system_count")
adef_system_count    = ad_mat.sum(axis=1).rename("adef_system_count")

print("defence_system_count per species (median):")
sp_series = arg_df.loc[included_ids, "species"]
print(defence_system_count.groupby(sp_series).median()
      .sort_values(ascending=False).round(1).to_string())
print()

# ── Ratio features ────────────────────────────────────────────────────────────
# denominator +1 prevents division by zero (a handful of genomes have 0 defence systems)
arg_unique = arg_df.loc[included_ids, "arg_count_unique"]
ime_unique = ime_df.loc[included_ids, "ime_count_unique"]

ratio_arg_defence  = (arg_unique  / (defence_system_count + 1)).rename("ratio_arg_defence")
ratio_ime_defence  = (ime_unique  / (defence_system_count + 1)).rename("ratio_ime_defence")
ratio_adef_defence = (adef_system_count / (defence_system_count + 1)).rename("ratio_adef_defence")

# ── Q2 target: within-species ARG burden tertile ─────────────────────────────
# pd.qcut divides a series into q equal-count bins.
# duplicates='drop' handles ties at tertile boundaries (common with integer counts).
tertile_labels = pd.Series(index=pd.Index(included_ids), dtype="object",
                            name="arg_burden_tertile")

for sp in SPECIES:
    sp_ids = sp_series[sp_series == sp].index
    sp_arg = arg_unique[sp_ids]
    try:
        labels = pd.qcut(
            sp_arg, q=3,
            labels=["low_ARG", "mid_ARG", "high_ARG"],
            duplicates="drop",
        )
    except ValueError:
        # Edge case: all values identical → assign mid_ARG to all
        labels = pd.Series("mid_ARG", index=sp_ids)
    tertile_labels[sp_ids] = labels.astype(str).values

print("ARG burden tertile — counts per species:")
print(pd.crosstab(sp_series, tertile_labels).to_string())
print()
print("Overall tertile totals:")
print(tertile_labels.value_counts().to_string())

defence_system_count per species (median):
species
kpneumoniae    22.0
ecloaceae      16.0
paeruginosa    14.0
saureus        12.0
efaecium        8.0
abaumannii      6.0

ARG burden tertile — counts per species:
arg_burden_tertile  high_ARG  low_ARG  mid_ARG
species                                       
abaumannii                42       59       49
ecloaceae                 47       50       49
efaecium                  44       60       46
kpneumoniae               42       44       46
paeruginosa                0        0      150
saureus                   50       56       44

Overall tertile totals:
arg_burden_tertile
mid_ARG     384
low_ARG     269
high_ARG    225


### Note: P. aeruginosa excluded from Q2 (ARG tertile failure)

P. aeruginosa triggered the ValueError fallback: all 150 genomes were assigned `mid_ARG`. This is not a code error.

**What happened:** The 33rd percentile of PA `arg_count_unique` equals the minimum value (5.0). 56 / 150 PA genomes (37%) have exactly ARG = 5 -- the species floor. `pd.qcut(q=3)` computes bin edges at the 0th, 33rd, 67th, and 100th percentiles: [5.0, 5.0, 8.0, 29.0]. The duplicate lower edge (0th == 33rd == 5.0) means only 2 distinct bins survive `duplicates='drop'`, and 2 bins cannot accept 3 labels.

**Biological meaning:** PA has insufficient ARG variance at the bottom of its distribution to define a `low_ARG` tertile that is biologically distinct from the middle. Any forced `low_ARG` bin would contain only genomes sitting at the species floor -- not a meaningful contrast.

**Consequence for Q2:** All PA genomes are `mid_ARG` and are dropped when Q2 filters for `low_ARG` vs `high_ARG`. Q2 will run on 5 species (728 genomes). Logged in `docs/decisions.md`.

In [25]:
# ── Step 4: assemble feature matrix ─────────────────────────────────────────
feature_matrix = pd.concat(
    [
        # Defence system blocks — add prefixes to make feature type explicit
        dp_mat.rename(columns=lambda c: f"dp_{c}"),
        dc_mat.rename(columns=lambda c: f"dc_{c}"),
        ad_mat.rename(columns=lambda c: f"ad_{c}"),

        # Per-tool count blocks
        arg_df.loc[included_ids,      ["arg_count_unique", "arg_count_total"]],
        ime_df.loc[included_ids,      ["ime_count_unique", "ime_count_total"]],
        amrfinder_df.loc[included_ids].drop(columns=["species"]),
        is_df_clean,

        # Summary and ratio features
        defence_system_count,
        adef_system_count,
        ratio_arg_defence,
        ratio_ime_defence,
        ratio_adef_defence,

        # Labels and metadata (not ML input — used for target and stratification)
        sp_series.rename("species"),
        tertile_labels,
        meta_df.loc[included_ids,  ["country", "year_bin", "complex_member"]],
        mlst_df.loc[included_ids,  ["sequence_type", "mlst_scheme"]],
    ],
    axis=1,
)

# ── Validation ───────────────────────────────────────────────────────────────
assert feature_matrix.shape[0] == len(included_ids), "Row count mismatch"
assert feature_matrix.index.duplicated().sum() == 0, "Duplicate genome IDs"

n_missing = feature_matrix.isnull().sum()
missing_cols = n_missing[n_missing > 0]

feature_cols = [c for c in feature_matrix.columns
                if c.startswith(("dp_", "dc_", "ad_", "arg_", "ime_",
                                  "hmrg_", "is_", "defence_", "adef_", "ratio_"))]
label_cols   = ["species", "arg_burden_tertile", "country", "year_bin",
                "complex_member", "sequence_type", "mlst_scheme"]

print(f"Feature matrix: {feature_matrix.shape[0]} genomes × {feature_matrix.shape[1]} columns")
print()
print(f"Feature columns: {len(feature_cols)}")
print(f"  dp_* (defence P/A):         {sum(c.startswith('dp_') for c in feature_cols)}")
print(f"  dc_* (defence counts):      {sum(c.startswith('dc_') for c in feature_cols)}")
print(f"  ad_* (anti-defence P/A):    {sum(c.startswith('ad_') for c in feature_cols)}")
print(f"  arg / ime / hmrg / is_:     {sum(c.startswith(('arg_','ime_','hmrg_','is_')) for c in feature_cols)}")
print(f"  summary + ratio:            {sum(c.startswith(('defence_','adef_','ratio_')) for c in feature_cols)}")
print(f"Label / metadata columns: {len(label_cols)}")
print()
if len(missing_cols) > 0:
    print("WARNING — missing values in:")
    print(missing_cols[missing_cols > 0].to_string())
else:
    print("Missing values in feature columns: none")
print()

# ── Save ─────────────────────────────────────────────────────────────────────
out_path = PROC / "feature_matrix.parquet"
feature_matrix.to_parquet(out_path)

fm_check = pd.read_parquet(out_path)
assert fm_check.shape == feature_matrix.shape

print(f"Saved → {out_path}")
print(f"File size: {out_path.stat().st_size / 1_000_000:.1f} MB")
print("Round-trip check passed.")

Feature matrix: 878 genomes × 631 columns

Feature columns: 625
  dp_* (defence P/A):         274
  dc_* (defence counts):      274
  ad_* (anti-defence P/A):    29
  arg / ime / hmrg / is_:     43
  summary + ratio:            5
Label / metadata columns: 7

WARNING — missing values in:
country           1
sequence_type    42



Saved → ../data/processed/feature_matrix.parquet
File size: 0.5 MB
Round-trip check passed.


### Section 9 comprehension check

Answer before moving to Section 10.

**Q1.** The ratio features use `(defence_system_count + 1)` as the denominator. Name two distinct failure modes that arise if you drop the `+1` and use `defence_system_count` directly.

**Q2.** K. pneumoniae has 132 genomes after exclusion. The tertile split uses `pd.qcut(q=3)`. How many genomes would you expect in the "low_ARG" tertile for KP — and why might the actual number differ slightly from that expectation?

**Q3.** After prefixing, the matrix has 274 `dp_*` + 274 `dc_*` + ≤29 `ad_*` ≈ 577 defence-related columns. For Phase 7 Random Forest with `max_features='sqrt'`, approximately how many of these 577 columns are considered at each tree split? What does this mean for the probability that a single defence system (e.g. RM_Type_I) influences any given split — and why does having both `dp_RM_Type_I` and `dc_RM_Type_I` in the matrix change that probability?